In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import re
import string
from sklearn.metrics.pairwise import cosine_similarity

from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

# CONFIG

In [ ]:
SEED = 42
NFOLDS = 5
MAX_LEN = 256//2
BATCH_SIZE = 16*2
EPOCHS = 5*2  # Updated to match nb1
MODEL_PATH = "/kaggle/input/deberta-v3-base/transformers/default/1/deberta-v3-base"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Ranking Loss Configuration
USE_RANKING_LOSS = True
RANKING_MARGIN = 0
BCE_WEIGHT = 0  # Weight for auxiliary BCE loss

# Semantic Similarity Configuration
USE_SEMANTIC_SIMILARITY = True  # Set to False to use random pairing (faster)
QWEN_MODEL_PATH = "Alibaba-NLP/gte-Qwen1.5-7B-instruct"  # Using 7B model from HuggingFace
TOP_K_NEGATIVES = 5  # Number of most similar negatives to pair with each positive

# Performance Notes:
# - Semantic similarity is slower but creates harder, more informative negative examples
# - Random pairing is faster but may create easier negative examples  
# - For quick testing, set USE_SEMANTIC_SIMILARITY = False
# - For best model performance, use semantic similarity

In [ ]:
# Set seeds - Updated to match nb1
import random
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.cuda.manual_seed_all(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# Load embedding model for semantic similarity
if USE_SEMANTIC_SIMILARITY:
    print("Loading Qwen embedding model...")
    embedding_model = SentenceTransformer(QWEN_MODEL_PATH, device=device)
    print(f"Embedding model loaded on {device}")

def clean_text(text):
    """Clean text for better embedding quality"""
    if pd.isna(text) or text == "":
        return ""
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '', text)
    
    # Remove email addresses
    text = re.sub(r'\S*@\S*\s?', '', text)
    
    # Remove excessive whitespace and newlines
    text = re.sub(r'\s+', ' ', text)
    
    # Remove special characters but keep basic punctuation
    text = re.sub(r'[^\w\s.,!?;:\'"()-]', '', text)
    
    # Remove extra spaces
    text = text.strip()
    
    return text

In [ ]:
# Load and preprocess data
train_path = "/kaggle/input/jigsaw-agile-community-rules/train.csv"
test_path = "/kaggle/input/jigsaw-agile-community-rules/test.csv"
sample_sub_path = "/kaggle/input/jigsaw-agile-community-rules/sample_submission.csv"

In [ ]:
df = pd.read_csv(train_path)
df["text"] = df["rule"] + " [SEP] " + df["body"]
df["label"] = df["rule_violation"].astype(float)

# Add improved data augmentation from nb1
def add_data(dataframe):
    ret=[[],[]]
    for i in ['positive_example_1','positive_example_2','negative_example_1','negative_example_2']:
        tmp= (dataframe['rule']+' [SEP] '+ dataframe[i]).tolist()
        ret[0]+= tmp
        ret[1]+= [1]*len(tmp) if 'positive' in i else [0]*len(tmp)
    return ret

In [ ]:
# Create augmented dataset with deduplication like nb1
test_df = pd.read_csv(test_path)

# Get augmented data from both df and test_df
augmented_train = add_data(df)
augmented_test = add_data(test_df)

# Combine original df with augmented data
augmented_texts = df.text.tolist() + augmented_train[0] + augmented_test[0]
augmented_labels = df.label.tolist() + augmented_train[1] + augmented_test[1]

# Create new augmented dataframe
augmented_df = pd.DataFrame({
    'text': augmented_texts,
    'label': augmented_labels
})
print(f'Before deduplication: {augmented_df.shape}')

# Deduplicate by grouping similar texts and averaging labels
augmented_df = augmented_df.groupby(augmented_df['text'].str.lower(), as_index=False).agg({
    'text': 'first',  # Keep the original case of the first occurrence
    'label': 'mean'   # Take mean of labels
})
print(f'After deduplication: {augmented_df.shape}')

# Extract rule and body for rule mapping
augmented_df['rule'] = augmented_df.text.apply(lambda x: x.split(' [SEP] ')[0])
augmented_df['body'] = augmented_df.text.apply(lambda x: x.split(' [SEP] ')[1])

# Create rule mapping like nb1
rule_map = {i:j for j,i in enumerate(augmented_df.rule.str.lower().unique())}
augmented_df['rule_id'] = augmented_df.rule.str.lower().map(rule_map)

print(f"Number of unique rules: {len(rule_map)}")
augmented_df.head()

In [ ]:
# Load tokenizer locally
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast = False)

In [ ]:
# Test semantic similarity pair selection (for verification only)
if USE_SEMANTIC_SIMILARITY and not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    print("=== Testing Semantic Similarity Pair Selection ===")
    
    # Create a small test dataset
    test_positives = [
        "This post is spam advertising a product for sale",
        "Clear violation of no advertising rule with promotional content",
        "Obvious commercial spam trying to sell something"
    ]
    
    test_negatives = [
        "Just sharing my experience with this topic",
        "This is completely unrelated political discussion",
        "Asking for help with technical issue",
        "Spam and advertising content here",  # This should be most similar to positives
        "Commercial promotion and sales pitch"  # This should also be similar
    ]
    
    print(f"Test positives: {len(test_positives)}")
    print(f"Test negatives: {len(test_negatives)}")
    
    if 'embedding_model' in globals():
        test_pairs = find_semantic_negative_pairs(test_positives, test_negatives, embedding_model, top_k=2)
        
        print(f"\n=== Semantic Similarity Results ===")
        for i, (pos, neg) in enumerate(test_pairs):
            print(f"Pair {i+1}:")
            print(f"  Positive: {pos[:60]}...")
            print(f"  Negative: {neg[:60]}...")
            print()
    else:
        print("Embedding model not loaded - skipping test")
    
    print("=== Test Complete ===\n")

In [ ]:
def create_rule_pairs(dataframe):
    """Create positive-negative pairs within each rule with sampling - Updated for augmented_df"""
    pairs = []
    rules = []
    targets = []
    
    # Group by rule
    for rule in dataframe['rule'].unique():
        rule_data = dataframe[dataframe['rule'] == rule]
        
        # Get positive and negative examples for this rule
        positives = rule_data[rule_data['label'] >= 0.5]['text'].tolist()
        negatives = rule_data[rule_data['label'] < 0.5]['text'].tolist()
        
        # Remove duplicates
        positives = list(set(positives))
        negatives = list(set(negatives))
        
        # Create all possible pairs for this rule
        rule_pairs = []
        for pos in positives:
            for neg in negatives:
                rule_pairs.append((pos, neg))
                
        MAX_PAIRS_PER_RULE = min(len(positives), len(negatives)) * 5
        
        # Sample pairs if too many
        if len(rule_pairs) > MAX_PAIRS_PER_RULE:
            np.random.seed(SEED)
            sampled_indices = np.random.choice(len(rule_pairs), MAX_PAIRS_PER_RULE, replace=False)
            rule_pairs = [rule_pairs[i] for i in sampled_indices]
        
        # Add sampled pairs
        pairs.extend(rule_pairs)
        rules.extend([rule] * len(rule_pairs))
        targets.extend([1] * len(rule_pairs))  # 1 means positive should rank higher than negative
        
        print(f"Rule '{rule}': {len(positives)} pos, {len(negatives)} neg -> {len(rule_pairs)} pairs")
    
    return pairs, rules, targets

In [ ]:
def create_rule_pairs(dataframe):
    """Create positive-negative pairs within each rule using semantic similarity"""
    pairs = []
    rules = []
    targets = []
    
    # Group by rule
    for rule in dataframe['rule'].unique():
        rule_data = dataframe[dataframe['rule'] == rule]
        
        # Get positive and negative examples for this rule
        positives = rule_data[rule_data['label'] >= 0.5]['text'].tolist()
        negatives = rule_data[rule_data['label'] < 0.5]['text'].tolist()
        
        # Remove duplicates
        positives = list(set(positives))
        negatives = list(set(negatives))
        
        print(f"\nRule '{rule[:50]}...': {len(positives)} pos, {len(negatives)} neg")
        
        if len(positives) == 0 or len(negatives) == 0:
            print(f"Skipping rule - insufficient pos/neg examples")
            continue
            
        # Choose pairing strategy
        if USE_SEMANTIC_SIMILARITY and 'embedding_model' in globals():
            print("Using semantic similarity for pair selection...")
            # Use semantic similarity to find hard negatives
            rule_pairs = find_semantic_negative_pairs(
                positives, negatives, embedding_model, top_k=TOP_K_NEGATIVES
            )
        else:
            print("Using random pairing (fallback)...")
            # Fallback to random pairing
            rule_pairs = []
            for pos in positives:
                for neg in negatives:
                    rule_pairs.append((pos, neg))
                    
            # Sample if too many pairs
            MAX_PAIRS_PER_RULE = min(len(positives), len(negatives)) * 5
            if len(rule_pairs) > MAX_PAIRS_PER_RULE:
                np.random.seed(SEED)
                sampled_indices = np.random.choice(len(rule_pairs), MAX_PAIRS_PER_RULE, replace=False)
                rule_pairs = [rule_pairs[i] for i in sampled_indices]
        
        # Add pairs to final list
        pairs.extend(rule_pairs)
        rules.extend([rule] * len(rule_pairs))
        targets.extend([1] * len(rule_pairs))  # 1 means positive should rank higher than negative
        
        print(f"Created {len(rule_pairs)} pairs for this rule")
    
    print(f"\nTotal pairs created: {len(pairs)}")
    return pairs, rules, targets

In [ ]:
class JigsawRankingDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.tokenizer = tokenizer
        self.max_len = max_len
        
        # Create rule-aware pairs using augmented dataframe
        self.pairs, self.rules, self.targets = create_rule_pairs(dataframe)
        print(f'Created {len(self.pairs)} ranking pairs from augmented data')

    def __len__(self): 
        return len(self.pairs)

    def __getitem__(self, idx):
        pos_text, neg_text = self.pairs[idx]
        target = self.targets[idx]  # Should be 1 for MarginRankingLoss
        
        # Tokenize positive example
        pos_enc = self.tokenizer(
            pos_text, padding='max_length', truncation=True, 
            max_length=self.max_len, return_tensors="pt"
        )
        
        # Tokenize negative example
        neg_enc = self.tokenizer(
            neg_text, padding='max_length', truncation=True, 
            max_length=self.max_len, return_tensors="pt"
        )
        
        return {
            'pos_input_ids': pos_enc['input_ids'].squeeze(0),
            'pos_attention_mask': pos_enc['attention_mask'].squeeze(0),
            'neg_input_ids': neg_enc['input_ids'].squeeze(0),
            'neg_attention_mask': neg_enc['attention_mask'].squeeze(0),
            'target': torch.tensor(target, dtype=torch.float),  # 1 for pos > neg
            'rule': self.rules[idx]
        }

# Updated dataset for validation with rule_id support like nb1
class JigsawDataset(Dataset):
    def __init__(self, texts, labels, rule_ids, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.rule_ids = rule_ids

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        enc = self.tokenizer(
            text, padding='max_length', truncation=True, max_length=self.max_len, return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        item['rule_ids'] = torch.tensor(self.rule_ids[idx])
        return item

In [ ]:
class JigsawModel(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.base = AutoModel.from_pretrained(model_path)
        self.drop = nn.Dropout(0.15)  # Match nb1's dropout
        self.out = nn.Linear(self.base.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.base(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        return self.out(self.drop(pooled)).squeeze(1)

In [ ]:
def train_one_epoch_ranking(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    ranking_criterion = nn.MarginRankingLoss(margin=RANKING_MARGIN)
    bce_criterion = nn.BCEWithLogitsLoss()
    
    for batch in tqdm(loader):
        optimizer.zero_grad()
        
        # Forward pass for positive examples
        pos_ids = batch["pos_input_ids"].to(DEVICE)
        pos_mask = batch["pos_attention_mask"].to(DEVICE)
        pos_logits = model(pos_ids, pos_mask)
        
        # Forward pass for negative examples
        neg_ids = batch["neg_input_ids"].to(DEVICE)
        neg_mask = batch["neg_attention_mask"].to(DEVICE)
        neg_logits = model(neg_ids, neg_mask)
        
        # Ranking targets (positive should rank higher than negative)
        targets = batch["target"].to(DEVICE)
        
        # Margin ranking loss
        ranking_loss = ranking_criterion(pos_logits, neg_logits, targets)
        
        # Optional: Add auxiliary BCE loss for regularization
        if BCE_WEIGHT > 0:
            # Treat positive examples as label 1, negative as label 0
            pos_bce = bce_criterion(pos_logits, torch.ones_like(pos_logits))
            neg_bce = bce_criterion(neg_logits, torch.zeros_like(neg_logits))
            bce_loss = (pos_bce + neg_bce) / 2
            total_loss_batch = ranking_loss + BCE_WEIGHT * bce_loss
        else:
            total_loss_batch = ranking_loss
        
        total_loss_batch.backward()
        # Add gradient clipping like nb1
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if scheduler:
            scheduler.step()
        total_loss += total_loss_batch.item()
    
    return total_loss / len(loader)

def train_one_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    for batch in tqdm(loader):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        logits = model(input_ids, mask)
        loss = nn.BCEWithLogitsLoss()(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if scheduler:
            scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)

In [ ]:
# Per-rule AUC validation function from nb1
def validate(model, loader):
    model.eval()
    preds, targets, rule_ids_list = [], [], []
    total_loss = 0
    criterion = nn.BCEWithLogitsLoss()
    
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            rule_ids = batch["rule_ids"]  # Assuming this is already on CPU as integers
            
            logits = model(input_ids, mask)
            loss = criterion(logits, labels)
            total_loss += loss.item()
            
            preds.extend(torch.sigmoid(logits).cpu().numpy())
            targets.extend(labels.cpu().numpy())
            rule_ids_list.extend(rule_ids.cpu().numpy() if torch.is_tensor(rule_ids) else rule_ids)
    
    # Convert to numpy arrays
    preds = np.array(preds)
    targets = np.array(targets)
    rule_ids_array = np.array(rule_ids_list)
    
    # Compute AUC per rule
    unique_rules = np.unique(rule_ids_array)
    rule_aucs = {}
    
    for rule_id in unique_rules:
        rule_mask = rule_ids_array == rule_id
        rule_preds = preds[rule_mask]
        rule_targets = targets[rule_mask]
        
        # Only compute AUC if we have both positive and negative samples for this rule
        if len(np.unique(rule_targets >= 0.5)) > 1:
            rule_auc = roc_auc_score(rule_targets >= 0.5, rule_preds)
            rule_aucs[rule_id] = rule_auc
        else:
            # If only one class present, we can't compute AUC
            rule_aucs[rule_id] = np.nan
    
    # Compute average AUC across rules (excluding NaN values)
    valid_aucs = [auc for auc in rule_aucs.values() if not np.isnan(auc)]
    avg_auc_per_rule = np.mean(valid_aucs) if valid_aucs else 0
    
    val_loss = total_loss / len(loader)
    return avg_auc_per_rule, val_loss, preds

def validate_simple(model, loader):
    """Simple validation on individual samples for ranking approach"""
    model.eval()
    preds, targets, rule_ids_list = [], [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            rule_ids = batch["rule_ids"]
            
            logits = model(input_ids, mask)
            
            preds.extend(torch.sigmoid(logits).cpu().numpy())
            targets.extend(labels.cpu().numpy())
            rule_ids_list.extend(rule_ids.cpu().numpy() if torch.is_tensor(rule_ids) else rule_ids)
    
    # Convert to numpy arrays
    preds = np.array(preds)
    targets = np.array(targets)
    rule_ids_array = np.array(rule_ids_list)
    
    # Compute AUC per rule like nb1
    unique_rules = np.unique(rule_ids_array)
    rule_aucs = {}
    
    for rule_id in unique_rules:
        rule_mask = rule_ids_array == rule_id
        rule_preds = preds[rule_mask]
        rule_targets = targets[rule_mask]
        
        if len(np.unique(rule_targets >= 0.5)) > 1:
            rule_auc = roc_auc_score(rule_targets >= 0.5, rule_preds)
            rule_aucs[rule_id] = rule_auc
        else:
            rule_aucs[rule_id] = np.nan
    
    # Compute average AUC across rules (excluding NaN values)
    valid_aucs = [auc for auc in rule_aucs.values() if not np.isnan(auc)]
    avg_auc_per_rule = np.mean(valid_aucs) if valid_aucs else 0
    
    return avg_auc_per_rule, np.array(preds)

In [ ]:
# Add linear scheduler import like nb1
from transformers import get_linear_schedule_with_warmup

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    all_preds = []
    folds = StratifiedKFold(n_splits=NFOLDS, shuffle=True, random_state=SEED)
    
    # Use augmented_df instead of df for stratification based on rule
    for fold, (tr_idx, val_idx) in enumerate(folds.split(augmented_df, augmented_df["rule"])):
        print(f"\n===== Fold {fold + 1} =====")
        
        if USE_RANKING_LOSS:
            # Prepare dataframes for ranking pairs (TRAINING ONLY)
            train_df = augmented_df.iloc[tr_idx].copy()
            
            # Create ranking training dataset
            train_ranking_ds = JigsawRankingDataset(train_df, tokenizer, MAX_LEN)
            train_ranking_loader = DataLoader(train_ranking_ds, batch_size=BATCH_SIZE//2, shuffle=True)
            
            # Individual validation dataset with rule_ids like nb1
            val_ds = JigsawDataset(
                augmented_df.iloc[val_idx]['text'].tolist(), 
                augmented_df.iloc[val_idx]['label'].tolist(), 
                augmented_df.iloc[val_idx]['rule_id'].tolist(),
                tokenizer, MAX_LEN
            )
            val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
            
        else:
            # Original approach with augmented data
            train_ds = JigsawDataset(
                augmented_df.iloc[tr_idx]['text'].tolist(), 
                augmented_df.iloc[tr_idx]['label'].tolist(),
                augmented_df.iloc[tr_idx]['rule_id'].tolist(),
                tokenizer, MAX_LEN
            )
            val_ds = JigsawDataset(
                augmented_df.iloc[val_idx]['text'].tolist(), 
                augmented_df.iloc[val_idx]['label'].tolist(),
                augmented_df.iloc[val_idx]['rule_id'].tolist(),
                tokenizer, MAX_LEN
            )
            train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
            val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    
        # Initialize model
        model = JigsawModel(MODEL_PATH).to(DEVICE)
        
        # Apply nb1's freezing strategy - only freeze embedding layers
        for name, param in model.named_parameters():
            if name.startswith('base.embedding'):
                param.requires_grad = False
        
        print('Trainable Params: ',sum(i.numel() for i in model.parameters() if i.requires_grad))
        
        # Use nb1's optimizer and scheduler configuration
        optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, eps=1e-6)
        
        if USE_RANKING_LOSS:
            total_steps = EPOCHS * len(train_ranking_loader)
        else:
            total_steps = EPOCHS * len(train_loader)
            
        warmup_steps = 0.1 * total_steps
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps,
        )
        
        best_auc = 0
        
        for epoch in range(EPOCHS):
            print(f"Epoch {epoch+1}/{EPOCHS}")
            
            if USE_RANKING_LOSS:
                # Train with ranking loss, validate with per-rule AUC
                train_loss = train_one_epoch_ranking(model, train_ranking_loader, optimizer, scheduler)
                val_auc, val_preds = validate_simple(model, val_loader)
                print(f"Train Loss: {train_loss:.4f}, Val AUC: {val_auc:.4f}")
            else:
                # Train with BCE loss
                train_loss = train_one_epoch(model, train_loader, optimizer, scheduler)
                val_auc, val_loss, val_preds = validate(model, val_loader)
                print(f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val AUC: {val_auc:.4f}")
            
            if val_auc > best_auc:
                best_auc = val_auc
                torch.save(model.state_dict(), f"model_fold{fold}_auc.bin")
    
        all_preds.append(pd.Series(val_preds))

In [ ]:
# Removed duplicate training loop - using updated version in cell-15

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    sample = pd.read_csv(sample_sub_path)
    df_test = pd.read_csv(test_path)
    df_test["text"] = df_test["rule"] + " [SEP] " + df_test["body"]
    
    test_preds = []
    for fold in range(NFOLDS):
        model = JigsawModel(MODEL_PATH).to(DEVICE)
        wts = torch.load(f"model_fold{fold}_auc.bin", map_location=DEVICE)
        wts = {k.replace('module.',''):v for k,v in wts.items()}
            
        model.load_state_dict(wts)
        model.eval()
    
        # Use simple dataset for test prediction (no rule_ids needed for test)
        test_ds = JigsawDataset(df_test['text'].tolist(), [0]*len(df_test), [0]*len(df_test), tokenizer, MAX_LEN)
        test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)
    
        fold_preds = []
        with torch.no_grad():
            for batch in test_loader:
                ids = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                logits = model(ids, mask)
                fold_preds.extend(torch.sigmoid(logits).cpu().numpy())
        test_preds.append(fold_preds)

In [ ]:
# Removed duplicate test inference - using updated version in cell-17

In [ ]:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    final_preds = np.mean(test_preds, axis=0)
    sample["rule_violation"] = final_preds
    sample.to_csv("submission.csv", index=False)
    print("✅ Submission saved as submission.csv")
else:
    !touch submission.csv
    
!head -n 4 submission.csv